# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SUKRIT004/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — Content lifecycle: The report compares growing and declining content and examines differences in performance across lifecycle states. This is relevant to my task because my model uses is_declining_label as the outcome. My methodology question is whether the label definition and observation window cleanly separate the information available before the outcome from the outcome itself. If the label is derived from trend_direction, then that field and related future information must not be used as model features.

Finding 2 — CTR and search position: The report identifies a relationship between CTR and search position, including a CTR difference across position groups. This supports my use of position and CTR as search-performance signals, but I would not interpret the relationship as causal. My methodology question is whether the comparison controls sufficiently for differences in page mix, exposure, and other factors. A grouped or time-aware validation design is therefore important when testing whether these signals generalize to unseen data.

These findings are useful as observed patterns, but they do not automatically establish that one variable causes another. My model validation should therefore focus on out-of-sample decision-support performance rather than treating the paper's observational relationships as causal proof.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I evaluated the Random Forest using a client-level holdout so that pages from the same client were not split across training and evaluation. The test set contained 6,163 rows from 7 held-out clients, while training contained 23,837 rows from 25 clients. On this split, the Random Forest achieved ROC-AUC of 0.6142 and Precision@50 of 0.82. The Week-4 baseline achieved Precision@50 of 0.84 on the same held-out rows. The observed result therefore favors the simpler baseline by 0.02 Precision@50 rather than the more complex model.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 — record the honest validation result from ML-08

# ML-09 Section 2 — Honest validation result

# ML-09 Section 2 — Honest validation result

import pandas as pd

validation_results = pd.DataFrame({
    "method": ["Week-4 baseline", "Random Forest"],
    "precision_at_50": [0.84, 0.82]
})

display(validation_results)

baseline = validation_results.loc[0, "precision_at_50"]
model = validation_results.loc[1, "precision_at_50"]

difference = baseline - model

print(f"Baseline Precision@50: {baseline:.2f}")
print(f"Random Forest Precision@50: {model:.2f}")
print(f"Baseline advantage: {difference:.2f}")

assert abs(difference - 0.02) < 1e-9

print("\n✓ Validation result recorded successfully.")

,method,precision_at_50
0,Week-4 baseline,0.84
1,Random Forest,0.82


Baseline Precision@50: 0.84
Random Forest Precision@50: 0.82
Baseline advantage: 0.02

✓ Validation result recorded successfully.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited the final feature set for direct outcome leakage. The target is_declining_label is derived from trend_direction, so is_declining_label, trend_direction, and trend_pct were excluded from the model features. content_id and client_id were also excluded because they are identifiers rather than predictive measurements. The final feature set therefore uses current search, traffic, engagement, content, and position signals without directly including the outcome.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 — Final feature leakage audit

# These are the exact features used in the Week-5 Random Forest.
model_features = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "search_volume",
    "competition",
    "cpc",
    "content_type",
    "main_intent",
    "competition_level",
    "age_tier",
    "freshness_tier",
    "position_tier",
    "impression_tier"
]

# Fields that must never be model features
forbidden = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

leaked_features = [
    feature for feature in model_features
    if feature in forbidden
]

print("Number of model features:", len(model_features))
print("Forbidden fields found:", leaked_features)

assert len(leaked_features) == 0, (
    f"Potential leakage detected: {leaked_features}"
)

print("\nLeakage audit PASSED.")
print("No target, trend-derived, or identifier fields were used as model features.")

Number of model features: 25
Forbidden fields found: []

Leakage audit PASSED.
No target, trend-derived, or identifier fields were used as model features.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

On the held-out client evaluation used in this experiment, the Random Forest showed some ability to distinguish observed declining pages, with a ROC-AUC of 0.6142 and Precision@50 of 0.82. The Week-4 baseline achieved a higher Precision@50 of 0.84 on the same evaluation set. These results are observed and measured for this validation setup and provide directional decision-support evidence; they do not establish causality or predict Google's ranking algorithm.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 4 — Claim rewrite

print("Original claim:")
print("The Random Forest predicts which pages will decline.")

print("\nSafe claim:")
print(
    "On the held-out client evaluation, the Random Forest showed "
    "some ability to distinguish observed declining pages, with "
    "ROC-AUC = 0.6142 and Precision@50 = 0.82. "
    "The Week-4 baseline achieved Precision@50 = 0.84 on the "
    "same evaluation set. These results are observed, measured, "
    "and directional for this validation setup and are intended "
    "as decision-support evidence, not causal proof or a prediction "
    "of Google's ranking algorithm."
)

Original claim:
The Random Forest predicts which pages will decline.

Safe claim:
On the held-out client evaluation, the Random Forest showed some ability to distinguish observed declining pages, with ROC-AUC = 0.6142 and Precision@50 = 0.82. The Week-4 baseline achieved Precision@50 = 0.84 on the same evaluation set. These results are observed, measured, and directional for this validation setup and are intended as decision-support evidence, not causal proof or a prediction of Google's ranking algorithm.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.